In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import warnings

import doralite
import gfdl_utils.core as gu
import CM4Xutils
import numpy as np
import xarray as xr
import xgcm

import gsw, xwmt, xhistogram
import zarr
from dask.diagnostics import ProgressBar

In [3]:
import cmocean
import matplotlib.colors as colors
import matplotlib.ticker as mtick
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.legend_handler import HandlerLine2D, HandlerTuple
plt.rcParams.update({'font.size': 12})

In [4]:
from common import *
grids = load_datasets()

sim = 'CM4Xp125_forced'
grid = grids[sim]
ds = grid._ds

No ideal age companion store for CM4Xp25 (missing ['CM4Xp25_historical_ideal_age_z.zarr', 'CM4Xp25_ssp585_ideal_age_z.zarr']); `agessc` will be absent from the CM4Xp25 dataset.
Inferring Z grid coordinate: depth `z_`


/work5/hfd/projects/CM4XTransientTracers/analysis/common.py:75: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.
  xr.open_zarr(f"../data/interim/{model}_historical_transient_tracers_surface.zarr"),


/work5/hfd/projects/CM4XTransientTracers/analysis/common.py:64: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.
  return xr.concat([xr.open_zarr(p) for p in paths], dim="year")


/work5/hfd/projects/CM4XTransientTracers/analysis/common.py:64: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.
  return xr.concat([xr.open_zarr(p) for p in paths], dim="year")


Inferring Z grid coordinate: depth `z_`


In [5]:
tracer = "cfc11"
ds["volcello"] = ds.thkcello.fillna(0.)*ds.areacello.fillna(0.)
tracer_content = ds[tracer].fillna(0.)*ds.volcello

### Layer thickness and CFC content

In [6]:
ds_layers = xr.Dataset(coords=ds.coords)
ds_layers = ds_layers.assign_coords({k:v for (k,v) in moc_metrics.coords.items() if "moc" in k})

ds_layers["tracer_binned"] = grid.transform(
    tracer_content,
    "Z",
    target=moc_metrics.sigma2_moc_i,
    target_data=grid.interp(ds.sigma2.ffill("z_l", limit=1), "Z"),
    method="conservative",
).rename({"rho2_moc_i":"rho2_moc_l"}).assign_coords({"rho2_moc_l":moc_metrics.rho2_moc_l})

ds_layers["layer_thickness"] = grid.transform(
    ds.thkcello.fillna(0.),
    "Z",
    target=moc_metrics.sigma2_moc_i,
    target_data=grid.interp(ds.sigma2.ffill("z_l", limit=1), "Z"),
    method="conservative",
).rename({"rho2_moc_i":"rho2_moc_l"}).assign_coords({"rho2_moc_l":moc_metrics.rho2_moc_l})

ds_layers.to_netcdf(f"../data/interim/tracer_content_by_layer_{sim}.nc", mode="w")

### Time mean over specific year ranges

In [7]:
ds_layers_processed = xr.Dataset(coords={k:v for (k,v) in ds.coords.items() if "time" not in k})
ds_layers_processed[f"{tracer}_layer"] = xr.concat(
    [
        (ds_layers["tracer_binned"] / ds_layers.areacello).fillna(0.).sel(year=slice(1991+years_after, 1997+years_after)).mean("year")
        .expand_dims({"year_start": np.array([1991+years_after])})
        for years_after in [0, 30, 60]
    ],
    dim="year_start"
)
ds_layers_processed[f"{tracer}_layer"].attrs = {
    "long_name": f"vertically-integrated {tracer} concentration",
    "units": "mol/m^2",
    "description": f"time-mean vertically-integrated {tracer} concentration from `year_start` to `year_start + 6` (inclusive)",
}
ds_layers_processed.to_netcdf(f"../data/processed/tracer_content_by_layer_intervalmeans_{sim}.nc", mode="w")